# WCAG 2.2 Violation Detection with Strands Decider

Train a Strands Decider model to classify HTML chunks as having WCAG 2.2 violations.

In [ ]:
# Clone the WCAG fork
!git clone https://github.com/turbolego/strands-decider-wcag.git
%cd strands-decider-wcag

In [ ]:
# Install dependencies
!pip install -e ".[train]" -q
!pip install torch flash-linear-attention -q

## Check GPU

In [ ]:
!nvidia-smi

## Check existing training data

In [ ]:
!ls -la data/wcag/

In [ ]:
import json
with open('data/wcag/train.jsonl') as f:
    rows = [json.loads(l) for l in f]
print(f'{len(rows)} training rows')

# Count rules
from collections import Counter
rules = Counter()
for r in rows:
    for q in r['questions']:
        rules[q] += 1
print('Rules:', dict(rules.most_common(10)))

## Stage 1: Build corpus (fast, CPU)

In [ ]:
# Build the WCAG training corpus
!bash training/recipe_v7.sh build

In [ ]:
# Verify corpus was built
!wc -l data/train_v5.jsonl

## Stage 2: Fetch upstream datasets (ContractNLI, MuSiQue)

In [ ]:
!bash training/recipe.sh fetch

## Stage 3: Generate synthetic WCAG data (GPU — slow)

In [ ]:
# Generate synthetic WCAG training data via Playwright + axe-core
# This creates additional training examples from real pages
!python scripts/generate_wcag_data.py \
  --url https://www.cdc.gov/ \
  --output data/wcag/synth_train.jsonl \
  --max-tokens 3000

## Stage 4: Multistep processing (GPU — slowest)

In [ ]:
# Run multistep + synthetic generators
!bash training/recipe.sh generated

## Stage 5: Teacher model distillation (GPU — ~1h)

In [ ]:
# Distill frozen Qwen3.5-4B teacher for multi-step rows
!bash training/recipe.sh teacher

## Stage 6: Parent model training (GPU — ~5h on P100)

In [ ]:
# Train parent model
export NGPU=1
!bash training/recipe.sh parent

## Stage 7: Replay parent for multi-step rows (GPU — ~40min)

In [ ]:
# Replay parent distributions for multi-step rows
!bash training/recipe.sh replay

## Stage 8: Final model training (GPU — ~6h on P100, 1h on H100)

In [ ]:
# Train final Strands Decider WCAG model
export NGPU=1
!bash training/recipe.sh train

## Stage 9: Calibrate confidence scores (GPU — fast)

In [ ]:
# Calibrate confidence scores on held-out sets
!bash training/recipe.sh calibrate

## Stage 10: Evaluation (GPU — fast)

In [ ]:
# Run evaluation on held-out sets
!bash training/recipe.sh eval

## Serve the trained model

In [ ]:
# Find the checkpoint
!find checkpoints/ -name '*.pt' -o -name '*.safetensors' | head -10

In [ ]:
# Serve the model
strands-decider serve checkpoints/hobson-2b-wcag --port 8000 &

## Test the model

In [ ]:
import requests, json

state = '''<button style="width: 20px; height: 20px; padding: 0;">Login</button>'''
questions = {
    'wcag_2_5_8_target_size': {
        'type': 'noul',
        'instructions': 'Does this button meet the WCAG 2.2 minimum target size of 24x24 CSS pixels?'
    }
}

resp = requests.post('http://localhost:8000/v1/systemone',
    json={'state': state, 'questions': questions},
    headers={'content-type': 'application/json'})
print(json.dumps(resp.json(), indent=2))

## Kaggle Session Management Notes

In [ ]:
# Kaggle has a 12-hour continuous session limit
# If training exceeds 12h, use checkpoint/resume:
# 1. The recipe saves checkpoints automatically in checkpoints/
# 2. To resume, set NGPU and re-run the failing stage
# 3. Use 'strands-decider train --config configs/train.yaml --resume checkpoints/hobson-2b-wcag'
# 4. Or use 'training/recipe.sh train' which checks for existing checkpoints